In [ ]:
# ============================================================================
# CELL 1 - Environment, imports, seed
# ============================================================================
import os, re, json, math, random, warnings
from pathlib import Path
from collections import Counter, defaultdict

import numpy as np
import pandas as pd
from tqdm.auto import tqdm

import torch
import torch.nn as nn
import torch.nn.functional as F
import torchaudio
from sklearn.model_selection import GroupKFold

warnings.filterwarnings("ignore")
SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED); torch.cuda.manual_seed_all(SEED)

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("torch     :", torch.__version__)
print("torchaudio:", torchaudio.__version__)
print("device    :", DEVICE, "| n_gpu:", torch.cuda.device_count())
for i in range(torch.cuda.device_count()):
    p = torch.cuda.get_device_properties(i)
    print(f"  GPU {i}: {p.name}  {p.total_memory/1e9:.1f} GB")

In [ ]:
# ============================================================================
# CELL 2 - Paths.  TRAIN + PUBLIC-TEST (public test ships gold transcripts, so
# we can compute the true competition score on it as a held-out dev set).
# Adjust the two roots to match your Kaggle dataset slugs.
# ============================================================================
TRAIN_ROOT = Path("/kaggle/input/mdd-dataset/MDD-Challenge-2025-training-set")
TEST_ROOT  = Path("/kaggle/input/mdd-dataset/MDD-Challenge-2025-public-test")

# fallback to local layout if the kaggle paths are absent
if not TRAIN_ROOT.exists():
    TRAIN_ROOT = Path("training/MDD-Challenge-2025-training-set")
if not TEST_ROOT.exists():
    TEST_ROOT = Path("testing/MDD-Challenge-2025-public-test")

TRAIN_META = TRAIN_ROOT / "metadata"
TEST_META  = TEST_ROOT  / "metadata"
WORK_DIR   = Path("/kaggle/working") if Path("/kaggle/working").exists() else Path("./work")
WORK_DIR.mkdir(parents=True, exist_ok=True)

for p in (TRAIN_META, TEST_META):
    assert p.exists(), f"missing path: {p}"
print("paths OK")
print("  train root:", TRAIN_ROOT)
print("  test  root:", TEST_ROOT)
print("  work  dir :", WORK_DIR)

In [ ]:
# ============================================================================
# CELL 3 - Load metadata (train + public test) + lexicon
# ============================================================================
def load_split(meta_dir, root):
    text   = pd.read_csv(meta_dir / [f for f in ["train.csv","public_test.csv"] if (meta_dir/f).exists()][0])
    phones = pd.read_csv(meta_dir / [f for f in ["train_phones.csv","public_test_phones.csv"] if (meta_dir/f).exists()][0])
    assert (text["id"].values == phones["id"].values).all(), "id ordering mismatch"
    return pd.DataFrame({
        "id":              text["id"],
        "path":            text["path"],
        "canonical_text":  text["canonical"],
        "transcript_text": text["transcript"],
        "canonical":       phones["canonical"],
        "transcript":      phones["transcript"],
    })

train_df = load_split(TRAIN_META, TRAIN_ROOT)
test_df  = load_split(TEST_META,  TEST_ROOT)
print("train:", train_df.shape, "| public-test:", test_df.shape)

lex = {}
with open(TRAIN_META / "lexicon_vmd.txt", encoding="utf-8") as f:
    for line in f:
        toks = line.rstrip("\n").split(maxsplit=1)
        if len(toks) == 2:
            lex[toks[0]] = toks[1].strip()
print("lexicon words:", len(lex))

In [ ]:
# ============================================================================
# CELL 4 - Tokenize phones, parse (base, tone), build inventory
# tone digit is the trailing -N (0..5); finals carry a 'z' suffix and no tone.
# ============================================================================
def tokenize_phones(s):
    if not isinstance(s, str): return []
    return [t for t in s.replace("$", " ").split() if t]

def base_tone(tok):
    if "-" in tok:
        b, t = tok.rsplit("-", 1)
        if t and t[0].isdigit():
            return b, t
    return tok, None

for d in (train_df, test_df):
    d["canon_ph"] = d["canonical"].apply(tokenize_phones)
    d["trans_ph"] = d["transcript"].apply(tokenize_phones)

canon_cnt = Counter(p for seq in train_df["canon_ph"] for p in seq)
trans_cnt = Counter(p for seq in train_df["trans_ph"] for p in seq)
test_ph   = set(p for seq in test_df["canon_ph"] for p in seq) | set(p for seq in test_df["trans_ph"] for p in seq)

PHONE_VOCAB = sorted(set(canon_cnt) | set(trans_cnt) | test_ph)
print("phone inventory (incl. test):", len(PHONE_VOCAB))
oov = test_ph - (set(canon_cnt) | set(trans_cnt))
print("phones in test but NOT in train:", sorted(oov))

In [ ]:
# ============================================================================
# CELL 5 - Vocabularies.  Main CTC: pad==blank==0, then real phones, then an
# ANTI-PHONE "!p" for every phone (Pillar 1).  A substituted phone is trained as
# its anti-phone, so the model is explicitly supervised to MARK a deviation
# instead of copying the canonical it is conditioned on (which is what collapses
# recall).  Anti-phones are a TRAINING device only: at decode "!p" maps back to
# the real phone "p" via ID2REAL, so the submission/grader never see anti-tokens.
# Tone CTC (auxiliary): 0=blank, 1=<none>, 2..7 = tones 0..5
# ============================================================================
PHONE2ID = {"<pad>": 0}
for i, ph in enumerate(PHONE_VOCAB):
    PHONE2ID[ph] = i + 1
N_REAL = len(PHONE2ID)                 # real ids: 0=<pad>/blank, 1..N_REAL-1 = phones
ANTI_OFFSET = N_REAL                   # anti id of a phone = its real id + ANTI_OFFSET
for ph in PHONE_VOCAB:
    PHONE2ID["!" + ph] = PHONE2ID[ph] + ANTI_OFFSET
ID2PHONE = {i: p for p, i in PHONE2ID.items()}
# decode map: every id (real OR anti) -> its underlying REAL phone string
ID2REAL = {i: (p[1:] if i >= ANTI_OFFSET else p) for p, i in PHONE2ID.items()}
VOCAB_SIZE = len(PHONE2ID)
BLANK_ID = 0
def is_anti(i): return i >= ANTI_OFFSET

TONES = ["0","1","2","3","4","5"]
TONE2ID = {"<blank>":0, "<none>":1}
for t in TONES: TONE2ID[t] = len(TONE2ID)
N_TONE = len(TONE2ID)

def enc_phones(seq): return [PHONE2ID[p] for p in seq]
def enc_tones(seq):
    out = []
    for p in seq:
        _, t = base_tone(p)
        out.append(TONE2ID["<none>"] if t is None else TONE2ID[t])
    return out

for d in (train_df, test_df):
    d["canon_ids"] = d["canon_ph"].apply(enc_phones)
    d["trans_ids"] = d["trans_ph"].apply(enc_phones)
    d["tone_ids"]  = d["trans_ph"].apply(enc_tones)

with open(WORK_DIR / "phoneme_vocab.json", "w", encoding="utf-8") as f:
    json.dump(PHONE2ID, f, ensure_ascii=False, indent=2)
print(f"VOCAB_SIZE={VOCAB_SIZE} ({N_REAL} real + {VOCAB_SIZE-N_REAL} anti, blank/pad=0) | N_TONE={N_TONE}")

In [ ]:
# ============================================================================
# CELL 6 - Speaker extraction + 5-fold speaker-grouped CV (no speaker leakage)
# ============================================================================
SPK_RE = re.compile(r"^(?P<spk>.+)_(?P<utt>\d+)$")
def extract_speaker(uid):
    m = SPK_RE.match(uid); return m.group("spk") if m else uid

train_df["speaker"] = train_df["id"].apply(extract_speaker)
print("unique train speakers:", train_df["speaker"].nunique())

N_FOLDS = 5
gkf = GroupKFold(n_splits=N_FOLDS)
train_df["fold"] = -1
for k, (_, vi) in enumerate(gkf.split(train_df, groups=train_df["speaker"])):
    train_df.iloc[vi, train_df.columns.get_loc("fold")] = k
print(train_df["fold"].value_counts().sort_index().to_dict())

In [ ]:
# ============================================================================
# CELL 7 - Phoneme alignment (Levenshtein backtrace) + gold-label derivation
# ============================================================================
def align_phones(ref, hyp):
    R, H = len(ref), len(hyp)
    dp = [[0]*(H+1) for _ in range(R+1)]
    bt = [[None]*(H+1) for _ in range(R+1)]
    for i in range(1, R+1): dp[i][0] = i; bt[i][0] = "D"
    for j in range(1, H+1): dp[0][j] = j; bt[0][j] = "I"
    for i in range(1, R+1):
        for j in range(1, H+1):
            same = ref[i-1] == hyp[j-1]
            ch = ((dp[i-1][j-1] + (0 if same else 1), "=" if same else "S"),
                  (dp[i-1][j] + 1, "D"),
                  (dp[i][j-1] + 1, "I"))
            dp[i][j], bt[i][j] = min(ch, key=lambda x: x[0])
    i, j, out = R, H, []
    while i > 0 or j > 0:
        op = bt[i][j]
        if op in ("=", "S"): out.append((op, ref[i-1], hyp[j-1])); i -= 1; j -= 1
        elif op == "D":      out.append(("D", ref[i-1], None));    i -= 1
        else:                out.append(("I", None, hyp[j-1]));    j -= 1
    return out[::-1]

def canon_view(canon, other):
    # project an alignment onto canonical positions: per canon phone -> (label, diag)
    # label: 0 correct, 1 mispronounced ; diag: the other-side phone or <DEL>
    labs, diags, ci = [], [], 0
    L = len(canon)
    aln = align_phones(canon, other)
    cur = [(0, canon[k]) for k in range(L)]  # default correct
    pos = -1
    for op, r, h in aln:
        if op == "I":       # insertion does not consume a canon slot
            continue
        pos += 1
        if pos >= L: break
        if op == "=":   cur[pos] = (0, r)
        elif op == "S": cur[pos] = (1, h)
        elif op == "D": cur[pos] = (1, "<DEL>")
    labs  = [c[0] for c in cur]
    diags = [c[1] for c in cur]
    return labs, diags

def edit_distance(a, b):
    return sum(1 for op, *_ in align_phones(a, b) if op != "=")

def build_anti_target(canon_ph, trans_ph):
    # CTC target = the realized transcript, but every phone that is a SUBSTITUTION
    # of its aligned canonical phone is emitted as its anti-phone (id + ANTI_OFFSET).
    # Deletions emit nothing; insertions stay normal (they do not affect detection
    # F1/DER in the metric, only PER).  Length == len(trans_ph).
    out = []
    for op, r, h in align_phones(canon_ph, trans_ph):
        if op == "D":
            continue
        pid = PHONE2ID[h]
        out.append(pid + ANTI_OFFSET if op == "S" else pid)
    return out

for d in (train_df, test_df):
    d["trans_anti_ids"] = [build_anti_target(c, t) for c, t in zip(d["canon_ph"], d["trans_ph"])]

# local self-test of the anti-phone target (pure-python; uses REAL in-vocab tokens)
_a, _b, _c = sorted(PHONE_VOCAB)[:3]                # 3 distinct, guaranteed in-vocab phones
_can = [_a, _a, _c]; _trn = [_b, _a, _c]           # pos0 substituted, pos1+2 correct
_tgt = build_anti_target(_can, _trn)
assert [ID2REAL[i] for i in _tgt] == _trn, "anti decode must recover the realized phones"
assert is_anti(_tgt[0]) and not is_anti(_tgt[1]) and not is_anti(_tgt[2]), "sub->anti, correct->real"
print("anti-target self-test OK | example:", [ID2PHONE[i] for i in _tgt])

In [ ]:
# ============================================================================
# CELL 8 - Unified MDD scorer.  ALL three metrics derive from one decoded
# sequence per utterance (detection, diagnosis, PER are mutually consistent).
#   F1  : detection of mispronounced canonical phones (positive class = 1)
#   DER : among True Rejections (gold=1 AND pred=1), fraction with wrong diag
#   PER : phoneme error rate of hyp vs gold transcript
#   Score = 0.5*F1 + 0.4*(1-DER) + 0.1*(1-PER)
# `records` = list of dicts with keys canon_ph, trans_ph, hyp_ph
# ============================================================================
def score_mdd(records, verbose=True):
    tp = fp = fn = 0
    der_n = der_e = 0
    per_err = per_tot = 0
    for r in records:
        canon, trans, hyp = r["canon_ph"], r["trans_ph"], r["hyp_ph"]
        g_lab, g_diag = canon_view(canon, trans)
        p_lab, p_diag = canon_view(canon, hyp)
        for gl, pl, gd, pd in zip(g_lab, p_lab, g_diag, p_diag):
            if gl == 1 and pl == 1: tp += 1
            elif gl == 0 and pl == 1: fp += 1
            elif gl == 1 and pl == 0: fn += 1
            if gl == 1 and pl == 1:           # True Rejection -> diagnosis judged
                der_n += 1
                if gd != pd: der_e += 1
        per_err += edit_distance(trans, hyp)
        per_tot += max(len(trans), 1)
    prec = tp/(tp+fp) if (tp+fp) else 0.0
    rec  = tp/(tp+fn) if (tp+fn) else 0.0
    f1   = 2*prec*rec/(prec+rec) if (prec+rec) else 0.0
    der  = der_e/der_n if der_n else 1.0
    per  = per_err/per_tot if per_tot else 1.0
    score = 0.5*f1 + 0.4*(1-der) + 0.1*(1-per)
    out = dict(f1=f1, precision=prec, recall=rec, der=der, per=per, score=score,
               tp=tp, fp=fp, fn=fn, tr=der_n)
    if verbose:
        print(f"  F1={f1:.4f} (P={prec:.3f} R={rec:.3f} tp/fp/fn={tp}/{fp}/{fn}) "
              f"DER={der:.4f} PER={per:.4f}  ==> SCORE={score:.4f}")
    return out

In [ ]:
# ============================================================================
# CELL 9 - Dataset + collator
# Target = transcript (what was actually said). Linguistic encoder is conditioned
# on canonical. Audio augmentation is label-preserving (Stage A).
# ============================================================================
import soundfile as sf
from transformers import Wav2Vec2FeatureExtractor

SR = 16000
HOP = 320           # wav2vec2 downsample factor (16 kHz -> 50 fps acoustic frames)
USE_PITCH = True    # fuse F0/pitch features (key for the tone-dominated test)
PITCH_DIM = 7       # [tone-shape, register, dreg, log-energy, voicing, zcr-creak, HNR]
BASE_MODEL = "nguyenvulebinh/wav2vec2-base-vietnamese-250h"
feat_ex = Wav2Vec2FeatureExtractor.from_pretrained(BASE_MODEL)

def load_wav(root, path):
    wav, sr = sf.read(str(Path(root)/path), dtype="float32", always_2d=False)
    if wav.ndim > 1: wav = wav.mean(axis=1)
    if sr != SR:
        wav = torchaudio.functional.resample(torch.from_numpy(wav), sr, SR).numpy()
    return wav

def _frame_sig(wav, n_frames, hop=HOP, win=400):
    out = np.zeros((n_frames, win), np.float32)
    for i in range(n_frames):
        seg = wav[i*hop:i*hop+win]
        out[i, :len(seg)] = seg
    return out

def pitch_features(wav):
    # Merged tone representation (~50 fps), interpolated + gate-fused by the model.
    # Speaker-RELATIVE (register-invariant) F0 + register-FREE tone shape attack the
    # train->test speaker shift; voice-quality (energy/voicing/creak/HNR) separates
    # the tone pairs that share an F0 contour but differ by glottalization. Returns (Fp, 7).
    t = torch.from_numpy(np.ascontiguousarray(wav)).float().unsqueeze(0)
    try:
        f0 = torchaudio.functional.detect_pitch_frequency(
            t, SR, frame_time=0.02, freq_low=60, freq_high=400).squeeze(0).numpy()
    except Exception:
        return np.zeros((max(1, len(wav)//HOP), PITCH_DIM), np.float32)
    n = len(f0); fr = _frame_sig(wav, n); win = fr.shape[1]
    voiced = (f0 > 1.0).astype(np.float32)
    logf0 = np.log(np.clip(f0, 1e-3, None))
    vm = voiced > 0
    mu = logf0[vm].mean() if vm.any() else logf0.mean()
    sd = (logf0[vm].std() if vm.any() else logf0.std()) + 1e-6
    reg = ((logf0 - mu) / sd) * voiced                         # speaker-relative register
    k = 7                                                      # ~150 ms baseline window
    padd = np.pad(reg, (k//2, k//2), mode="edge")
    base = np.convolve(padd, np.ones(k)/k, mode="valid")[:len(reg)]
    shape = (reg - base).astype(np.float32)                    # register-free tone shape
    dreg = np.concatenate([[0.0], np.diff(reg)]).astype(np.float32)
    energy = np.log(np.maximum((fr**2).mean(1), 1e-8))
    energy = (energy - energy.mean()) / (energy.std() + 1e-6)
    zcr = (np.abs(np.diff(np.sign(fr), axis=1)) > 0).mean(1).astype(np.float32)
    zcr = (zcr - zcr.mean()) / (zcr.std() + 1e-6)              # creak -> high+irregular
    hnr = np.zeros(n, np.float32)
    for i in range(n):
        s = fr[i] - fr[i].mean(); e0 = float((s*s).sum()) + 1e-8
        ac = np.correlate(s, s, mode="full")[win-1:]
        lo, hi = 32, min(267, len(ac)-1)                       # ~60..500 Hz lags
        hnr[i] = ac[lo:hi].max()/e0 if hi > lo else 0.0
    return np.stack([shape, reg.astype(np.float32), dreg, energy.astype(np.float32),
                     voiced, zcr, hnr], axis=1).astype(np.float32)

class MDDDataset(torch.utils.data.Dataset):
    def __init__(self, frame, root, max_sec=14.0, augment=False):
        self.r = frame.reset_index(drop=True); self.root = root
        self.max_n = int(max_sec*SR); self.augment = augment
    def __len__(self): return len(self.r)
    def _aug(self, wav):
        if np.random.rand() < 0.5:          # additive noise 15-35 dB SNR
            snr = np.random.uniform(15, 35)
            sp = (wav**2).mean() + 1e-10
            wav = wav + np.random.randn(*wav.shape).astype(np.float32)*np.sqrt(sp/(10**(snr/10)))
        if np.random.rand() < 0.5:          # gain +-6 dB
            wav = wav * (10**(np.random.uniform(-6,6)/20))
        if np.random.rand() < 0.3:          # time shift +-100 ms
            s = int(np.random.uniform(-0.1,0.1)*SR)
            if s>0:   wav = np.concatenate([np.zeros(s,np.float32), wav[:-s]])
            elif s<0: wav = np.concatenate([wav[-s:], np.zeros(-s,np.float32)])
        return wav.astype(np.float32)
    def __getitem__(self, i):
        row = self.r.iloc[i]
        wav = load_wav(self.root, row["path"])
        if self.augment: wav = self._aug(wav)
        if len(wav) > self.max_n: wav = wav[:self.max_n]
        pitch = pitch_features(wav) if USE_PITCH else None   # computed AFTER augmentation
        return dict(idx=i, input_values=wav, pitch=pitch,
                    labels=row["trans_anti_ids"], tones=row["tone_ids"],   # anti-phone target
                    canon=row["canon_ids"])

class Collator:
    def __init__(self, fe): self.fe = fe
    def __call__(self, batch):
        a = self.fe([b["input_values"] for b in batch], sampling_rate=SR,
                    padding=True, return_tensors="pt", return_attention_mask=True)
        def pad2d(seqs, val=0):
            m = max(len(s) for s in seqs)
            t = torch.full((len(seqs), m), val, dtype=torch.long)
            ln = torch.tensor([len(s) for s in seqs], dtype=torch.long)
            for i, s in enumerate(seqs): t[i,:len(s)] = torch.tensor(s, dtype=torch.long)
            return t, ln
        labels, label_len = pad2d([b["labels"] for b in batch])
        tones,  tone_len  = pad2d([b["tones"]  for b in batch])
        canon,  canon_len = pad2d([b["canon"]  for b in batch])
        canon_pad = torch.arange(canon.size(1))[None,:] >= canon_len[:,None]  # True=pad
        out = dict(input_values=a["input_values"], attention_mask=a["attention_mask"],
                   labels=labels, label_len=label_len, tones=tones, tone_len=tone_len,
                   canon=canon, canon_pad=canon_pad,
                   idx=torch.tensor([b["idx"] for b in batch]))
        if batch[0]["pitch"] is not None:                    # pad pitch to (B, Fp_max, C)
            fp = max(p["pitch"].shape[0] for p in batch); C = batch[0]["pitch"].shape[1]
            pt = torch.zeros(len(batch), fp, C, dtype=torch.float32)
            for i, p in enumerate(batch):
                arr = torch.from_numpy(p["pitch"]); pt[i, :arr.shape[0]] = arr
            out["pitch"] = pt
        return out

collator = Collator(feat_ex)
print("dataset + collator ready")

In [ ]:
# ============================================================================
# CELL 10 - Linguistic-fused Wav2Vec2 for MDD (LingWav2Vec2-style) + Focal CTC
#   cross-attention: query = acoustic, key/value = canonical phoneme embeddings
#   RMSNorm + SwiGLU FFN ; main CTC head (121-phone) + auxiliary tone CTC head
# ============================================================================
from transformers import Wav2Vec2Model

class RMSNorm(nn.Module):
    def __init__(self, d, eps=1e-6):
        super().__init__(); self.w = nn.Parameter(torch.ones(d)); self.eps = eps
    def forward(self, x):
        return self.w * x * torch.rsqrt(x.pow(2).mean(-1, keepdim=True) + self.eps)

class SwiGLU(nn.Module):
    def __init__(self, d, h, p=0.1):
        super().__init__()
        self.w1 = nn.Linear(d, h); self.w2 = nn.Linear(d, h)
        self.w3 = nn.Linear(h, d); self.drop = nn.Dropout(p)
    def forward(self, x): return self.w3(self.drop(F.silu(self.w1(x)) * self.w2(x)))

class PosEnc(nn.Module):
    def __init__(self, d, mx=256):
        super().__init__()
        pe = torch.zeros(mx, d); pos = torch.arange(mx).unsqueeze(1).float()
        div = torch.exp(torch.arange(0, d, 2).float()*(-math.log(10000.0)/d))
        pe[:,0::2] = torch.sin(pos*div); pe[:,1::2] = torch.cos(pos*div)
        self.register_buffer("pe", pe.unsqueeze(0))
    def forward(self, x): return x + self.pe[:, :x.size(1)]

class LinguisticEncoder(nn.Module):
    def __init__(self, d, vocab, heads=8, ffn=1536, p=0.1):
        super().__init__()
        self.emb = nn.Embedding(vocab, d, padding_idx=0)
        self.pos = PosEnc(d)
        self.qn  = RMSNorm(d)
        self.attn = nn.MultiheadAttention(d, heads, dropout=p, batch_first=True)
        self.ffn_norm = RMSNorm(d); self.ffn = SwiGLU(d, ffn, p)
        self.out_norm = RMSNorm(d)
    def forward(self, acoustic, canon_ids, canon_pad):
        ling = self.pos(self.emb(canon_ids))                 # (B,L,d)
        att, _ = self.attn(self.qn(acoustic), ling, ling, key_padding_mask=canon_pad)
        x = acoustic + att
        x = x + self.ffn(self.ffn_norm(x))
        return self.out_norm(x)

class LingWav2Vec2MDD(nn.Module):
    def __init__(self, base, vocab, n_tone, use_tone=True, use_pitch=True, pitch_dim=2):
        super().__init__()
        self.w2v2 = Wav2Vec2Model.from_pretrained(base)
        self.w2v2.config.mask_time_prob = 0.05    # SpecAugment (active in train mode)
        self.w2v2.config.mask_feature_prob = 0.01
        self.w2v2.config.mask_time_length = 10
        self.w2v2.config.mask_feature_length = 16
        d = self.w2v2.config.hidden_size
        self.use_pitch = use_pitch
        if use_pitch:
            self.pitch_proj = nn.Sequential(nn.Linear(pitch_dim, d), nn.GELU(), nn.Linear(d, d))
            self.pitch_gate = nn.Parameter(torch.zeros(1))   # start at 0 -> no disruption
        self.ling = LinguisticEncoder(d, vocab)
        self.drop = nn.Dropout(0.1)
        self.lm_head = nn.Linear(d, vocab)
        self.use_tone = use_tone
        if use_tone: self.tone_head = nn.Linear(d, n_tone)
    def freeze_feature_encoder(self):
        self.w2v2.feature_extractor._freeze_parameters()
    def feat_lengths(self, attn_mask):
        return self.w2v2._get_feat_extract_output_lengths(attn_mask.sum(-1)).long()
    def forward(self, input_values, attention_mask, canon, canon_pad, pitch=None):
        h = self.w2v2(input_values, attention_mask=attention_mask).last_hidden_state
        if self.use_pitch and pitch is not None:
            p = pitch.transpose(1, 2)                                    # (B,C,Fp)
            p = F.interpolate(p, size=h.size(1), mode="linear", align_corners=False)
            p = self.pitch_proj(p.transpose(1, 2))                       # (B,T,d)
            h = h + torch.tanh(self.pitch_gate) * p                      # gated fusion
        h = self.ling(h, canon, canon_pad)
        h = self.drop(h)
        logits = self.lm_head(h)
        tone = self.tone_head(h) if self.use_tone else None
        return logits, tone

def focal_ctc(logits, targets, in_len, tgt_len, blank=0, alpha=0.99, gamma=2.0):
    lp = F.log_softmax(logits.float(), dim=-1).transpose(0, 1)   # (T,B,V)
    ctc = F.ctc_loss(lp, targets, in_len, tgt_len, blank=blank,
                     reduction="none", zero_infinity=True)       # per-sample NLL (sum)
    # focal weight uses a per-phone confidence p in (0,1); the loss MAGNITUDE stays the
    # raw NLL so gradients don't vanish (this is what breaks all-blank collapse).
    p = torch.exp(-(ctc / tgt_len.clamp(min=1).float()))
    return (alpha * (1 - p)**gamma * ctc).mean()

def plain_ctc(logits, targets, in_len, tgt_len, blank=0):
    lp = F.log_softmax(logits.float(), dim=-1).transpose(0, 1)
    return F.ctc_loss(lp, targets, in_len, tgt_len, blank=blank,
                      reduction="mean", zero_infinity=True)
print("model defined")

In [ ]:
# ============================================================================
# CELL 11 - Greedy CTC decode + evaluate a dataframe with the unified scorer
# ============================================================================
def ctc_greedy(logits, blank=0):
    pred = logits.argmax(-1).cpu().numpy()
    out = []
    for row in pred:
        prev, seq = -1, []
        for t in row:
            if t != prev and t != blank: seq.append(int(t))
            prev = t
        out.append(seq)
    return out

@torch.no_grad()
def evaluate_df(model, frame, root, batch=8, verbose=True, tag=""):
    model.eval()
    ds = MDDDataset(frame, root, augment=False)
    dl = torch.utils.data.DataLoader(ds, batch_size=batch, shuffle=False,
                                     collate_fn=collator, num_workers=2)
    recs = []
    fr = frame.reset_index(drop=True)
    for b in tqdm(dl, desc=f"eval {tag}", leave=False):
        pf = b.get("pitch"); pf = pf.to(DEVICE) if pf is not None else None
        logits, _ = model(b["input_values"].to(DEVICE), b["attention_mask"].to(DEVICE),
                          b["canon"].to(DEVICE), b["canon_pad"].to(DEVICE), pitch=pf)
        for j, ids in zip(b["idx"].tolist(), ctc_greedy(logits)):
            row = fr.iloc[j]
            recs.append(dict(canon_ph=row["canon_ph"], trans_ph=row["trans_ph"],
                             hyp_ph=[ID2REAL[i] for i in ids]))   # anti-phone -> real phone
    if verbose: print(f"[{tag}] n={len(recs)}")
    return score_mdd(recs, verbose=verbose), recs

In [ ]:
# ============================================================================
# CELL 12 - Train on the FULL train set, validate on the PUBLIC TEST.
# Multi-GPU via DataParallel (Kaggle 2x T4): each batch is split across GPUs.
# We train on ALL labeled train data (no held-out fold) to keep every one of
# the scarce error examples, and select the best checkpoint by the public-test
# competition score (matches the eval distribution). The public test is small
# (~309 mispron positions) and the ranking is on a HIDDEN set, so avoid heavy
# hyperparameter tuning against it - select epochs only.
# ============================================================================
from torch.cuda.amp import autocast, GradScaler

def make_parallel(m):
    if torch.cuda.device_count() > 1:
        return nn.DataParallel(m)
    return m

def train_full(epochs=30, bs=16, accum=1, lr=3e-5, head_lr=5e-4, warmup=0.05,
               tone_w=0.3, focal_gamma=2.0, freeze_cnn=True, out_name="full",
               val_frame=None, val_root=None, extra_df=None, extra_root=None,
               init_from=None, warmup_plain=1):
    if val_frame is None: val_frame = test_df
    if val_root  is None: val_root  = TEST_ROOT
    tr = train_df.reset_index(drop=True)
    ngpu = max(1, torch.cuda.device_count())
    base_ds = MDDDataset(tr, TRAIN_ROOT, augment=True)
    if extra_df is not None and len(extra_df):        # Stage B: mix in synthetic errors
        ds = torch.utils.data.ConcatDataset([base_ds, MDDDataset(extra_df, extra_root, augment=True)])
        print(f"train_full: train={len(tr)} + synth={len(extra_df)} = {len(ds)} | "
              f"val(public-test)={len(val_frame)} | GPUs={ngpu} | batch={bs} ({bs//ngpu}/GPU)")
    else:
        ds = base_ds
        print(f"train_full: train={len(tr)} | val(public-test)={len(val_frame)} | GPUs={ngpu} "
              f"| per-step batch={bs} (={bs//ngpu}/GPU) x accum {accum}")
    dl = torch.utils.data.DataLoader(ds, batch_size=bs, shuffle=True, drop_last=True,
                                     collate_fn=collator, num_workers=4, pin_memory=True)

    core = LingWav2Vec2MDD(BASE_MODEL, VOCAB_SIZE, N_TONE,
                           use_tone=tone_w>0, use_pitch=USE_PITCH, pitch_dim=PITCH_DIM).to(DEVICE)
    if init_from is not None:                          # warm-start (e.g. fine-tune Stage A)
        core.load_state_dict(torch.load(init_from, map_location=DEVICE)); print("warm-start:", init_from)
    if freeze_cnn: core.freeze_feature_encoder()
    model = make_parallel(core)                       # DataParallel wrapper (or core)

    # Differential LR: NEW randomly-initialized modules learn fast (head_lr); the
    # pretrained wav2vec2 backbone adapts slowly (lr). This is what gets CTC off the
    # all-blank plateau without wrecking the pretrained acoustics.
    new_keys = ("ling", "lm_head", "tone_head", "pitch_proj", "pitch_gate")
    head_p, base_p = [], []
    for n, p in core.named_parameters():
        if not p.requires_grad: continue
        (head_p if any(k in n for k in new_keys) else base_p).append(p)
    print(f"param groups: head={sum(p.numel() for p in head_p)/1e6:.1f}M @ {head_lr:.0e} | "
          f"backbone={sum(p.numel() for p in base_p)/1e6:.1f}M @ {lr:.0e}")
    opt = torch.optim.AdamW([{"params": base_p, "lr": lr},
                             {"params": head_p, "lr": head_lr}], weight_decay=0.005)
    total = (len(dl)//accum)*epochs; wu = int(warmup*total)
    def lr_lambda(s):
        if s < wu: return s/max(1,wu)
        prog = (s-wu)/max(1,total-wu); return 0.5*(1+math.cos(math.pi*prog))
    sched = torch.optim.lr_scheduler.LambdaLR(opt, lr_lambda)
    scaler = GradScaler()

    best = {"score": -1}
    for ep in range(epochs):
        model.train(); opt.zero_grad(); running = 0.0
        for st, b in enumerate(tqdm(dl, desc=f"ep{ep}", leave=False)):
            iv = b["input_values"].to(DEVICE); am = b["attention_mask"].to(DEVICE)
            cn = b["canon"].to(DEVICE); cp = b["canon_pad"].to(DEVICE)
            pf = b.get("pitch"); pf = pf.to(DEVICE) if pf is not None else None
            in_len = core.feat_lengths(am).to(DEVICE)   # helper lives on the unwrapped core
            with autocast():
                logits, tone = model(iv, am, cn, cp, pitch=pf)   # DP splits batch across both T4s
            # plain CTC for the first warmup_plain epoch(s) so the fresh (now larger,
            # anti-phone) head settles before focal up-weights the hard/rare phones.
            if ep < warmup_plain:
                loss = plain_ctc(logits, b["labels"].to(DEVICE), in_len, b["label_len"].to(DEVICE))
            else:
                loss = focal_ctc(logits, b["labels"].to(DEVICE), in_len,
                                 b["label_len"].to(DEVICE), gamma=focal_gamma)
            if tone is not None:
                loss = loss + tone_w*plain_ctc(tone, b["tones"].to(DEVICE), in_len,
                                               b["tone_len"].to(DEVICE))
            scaler.scale(loss/accum).backward()
            running += loss.item()
            if (st+1) % accum == 0:
                scaler.unscale_(opt); nn.utils.clip_grad_norm_(core.parameters(), 5.0)
                scaler.step(opt); scaler.update(); opt.zero_grad(); sched.step()
        agg, _ = evaluate_df(model, val_frame, val_root, batch=2*bs, tag=f"public ep{ep}")
        print(f"ep{ep} loss={running/len(dl):.3f} lr={sched.get_last_lr()[0]:.2e} "
              f"public_score={agg['score']:.4f} (P={agg['precision']:.3f} R={agg['recall']:.3f})")
        if agg["score"] > best["score"]:
            best = {"score": agg["score"], "ep": ep}
            torch.save(core.state_dict(), WORK_DIR / f"{out_name}.pt")   # unwrapped keys
            print(f"  * saved best (score={agg['score']:.4f})")
    print("best:", best)
    return core, best

In [ ]:
# ============================================================================
# CELL TONE-run - train Stage A with the merged prosody + voice-quality features.
# The auxiliary tone head now sees register-relative F0 + glottalization cues,
# which is exactly what the span override/diagnosis-prior pool at decode time.
# Bump epochs for the real run. (To stack on synthetic errors, run the main
# notebook's Stage B with these same features; here we isolate the tone subsystem.)
# ============================================================================
core, best = train_full(epochs=40, bs=16, out_name="exp_tone")
core.load_state_dict(torch.load(WORK_DIR / "exp_tone.pt", map_location=DEVICE))
model = core
print("loaded best exp_tone checkpoint | greedy public-test:", best)

In [ ]:
# ============================================================================
# CELL 14 - TRUE competition score on the labeled PUBLIC TEST with the best
# checkpoint. This is the number that matters - it matches the eval distribution
# (33.8% error utterances, ~72% of subs are tone errors). Watch P vs R:
# low recall -> add more error augmentation (Stage B); low precision -> ease it.
# ============================================================================
eval_model = make_parallel(model)   # use both T4s for inference too
print("=== PUBLIC TEST (true competition score) ===")
test_agg, test_recs = evaluate_df(eval_model, test_df, TEST_ROOT, batch=32, tag="publictest")

In [ ]:
# ============================================================================
# CELL 15 - CTC forced alignment (per canonical phone frame spans + GOP).
# Pure-PyTorch Viterbi over the blank-padded target lattice.
# Used by Stage-B augmentation (CELL 16) and optional GOP confidence gating.
# ============================================================================
@torch.no_grad()
def utter_logprobs(core, root, path, canon_ids):
    # run the (unwrapped) model with the canonical prompt; return wav + per-frame log-probs
    core.eval()
    wav = load_wav(root, path)
    a = feat_ex(wav, sampling_rate=SR, return_tensors="pt", return_attention_mask=True)
    iv = a["input_values"].to(DEVICE); am = a["attention_mask"].to(DEVICE)
    cn = torch.tensor([list(canon_ids)], device=DEVICE)
    cp = torch.zeros((1, len(canon_ids)), dtype=torch.bool, device=DEVICE)
    pf = None
    if USE_PITCH:
        pf = torch.from_numpy(pitch_features(wav)).unsqueeze(0).to(DEVICE)
    logits, _ = core(iv, am, cn, cp, pitch=pf)
    return wav, torch.log_softmax(logits[0].float(), dim=-1).cpu()

def ctc_forced_align(log_probs, target_ids, blank=0):
    T, V = log_probs.shape; L = len(target_ids)
    ext = [blank]
    for t in target_ids: ext += [t, blank]
    E = len(ext); ext_t = torch.tensor(ext)
    emit = log_probs[:, ext_t]; NEG = -1e18
    dp = torch.full((T, E), NEG); bp = torch.zeros((T, E), dtype=torch.long)
    dp[0,0] = emit[0,0]
    if E > 1: dp[0,1] = emit[0,1]
    for t in range(1, T):
        prev = dp[t-1]
        v_stay = prev
        v_step = torch.cat([torch.tensor([NEG]), prev[:-1]])
        v_skip = torch.cat([torch.tensor([NEG, NEG]), prev[:-2]])
        for s in range(E):
            if not (ext[s] != blank and s >= 2 and ext[s] != ext[s-2]):
                v_skip[s] = NEG
        stack = torch.stack([v_stay, v_step, v_skip], 0)
        bv, bi = stack.max(0); dp[t] = bv + emit[t]; bp[t] = bi
    end = E-1 if dp[T-1, E-1] >= dp[T-1, E-2] else E-2
    path = [0]*T; path[T-1] = end
    for t in range(T-1, 0, -1):
        s = path[t]; b = int(bp[t, s]); path[t-1] = s - (0 if b==0 else (1 if b==1 else 2))
    frames = [[] for _ in range(L)]
    for t, s in enumerate(path):
        if ext[s] != blank: frames[(s-1)//2].append(t)
    out = []
    for i, fr in enumerate(frames):
        if fr: out.append((fr[0], fr[-1], log_probs[fr, target_ids[i]].mean().item()))
        else:  out.append((None, None, -1e9))
    return out
print("forced-align ready")

In [ ]:
# ============================================================================
# CELL TONE - SPAN-POOLED TONE OVERRIDE + CONFUSION-PRIOR DIAGNOSIS (the experiment).
# (1) main CTC head -> base hypothesis (carries PER + segmental detection);
# (2) force-align canonical -> per-canon-phone frame span;
# (3) pool the tone-head posteriors over each tone-bearing canonical span;
# (4) re-rank tones by  pooled_posterior * P(realized|canonical)**beta  (prior from
#     TRAIN tone-only subs), then overwrite the rime's tone digit IF the winner beats
#     the hypothesis's CURRENT tone by margin m (gate that protects clean speech).
# Cache once, sweep (beta, m), pick by bootstrap. beta=0,m=0 == pure span override.
# ============================================================================
import numpy as np, pandas as pd
from collections import defaultdict
TONE_COLS = [TONE2ID[t] for t in TONES]            # posterior columns for tones 0..5

# --- tone-confusion prior P(realized | canonical) from TRAIN tone-only subs (no dev leak) ---
conf = defaultdict(lambda: np.ones(6, np.float64))  # +1 Laplace smoothing
for canon, trans in zip(train_df["canon_ph"], train_df["trans_ph"]):
    for op, r, h in align_phones(canon, trans):
        if op == "S" and r is not None and h is not None:
            rb, rt = base_tone(r); hb, ht = base_tone(h)
            if rt is not None and ht is not None and rb == hb:
                conf[rt][int(ht)] += 1.0
PRIOR_MAT = {}
for c in TONES:
    v = conf[c].copy(); v[int(c)] = max(v[int(c)], v.max())   # never penalize staying correct
    PRIOR_MAT[int(c)] = v / v.sum()
print("tone confusion prior (rows=canonical tone, cols=realized 0..5):")
for c in TONES: print(f"  {c}:", PRIOR_MAT[int(c)].round(3))

@torch.no_grad()
def cache_tone(core, frame, root):
    core.eval(); fr = frame.reset_index(drop=True); cache = []
    for _, row in tqdm(fr.iterrows(), total=len(fr), desc="cache tone posteriors"):
        wav = load_wav(root, row["path"])
        a = feat_ex(wav, sampling_rate=SR, return_tensors="pt", return_attention_mask=True)
        iv = a["input_values"].to(DEVICE); am = a["attention_mask"].to(DEVICE)
        cn = torch.tensor([list(row["canon_ids"])], device=DEVICE)
        cp = torch.zeros((1, len(row["canon_ids"])), dtype=torch.bool, device=DEVICE)
        pf = torch.from_numpy(pitch_features(wav)).unsqueeze(0).to(DEVICE) if USE_PITCH else None
        logits, tone = core(iv, am, cn, cp, pitch=pf)
        lp = torch.log_softmax(logits[0].float(), -1).cpu()
        tps = torch.softmax(tone[0].float(), -1).cpu().numpy()        # (T, N_TONE)
        hyp_ids, prev = [], -1
        for t in lp.argmax(-1).tolist():
            if t != prev and t != BLANK_ID: hyp_ids.append(t)
            prev = t
        hyp_ph = [ID2REAL[i] for i in hyp_ids]                        # anti-phone -> real phone
        spans = ctc_forced_align(lp, list(row["canon_ids"]))
        aln = align_phones(row["canon_ph"], hyp_ph); ci = hi = -1; h2c = {}
        for op, r, h in aln:
            if op in ("=", "S"): ci += 1; hi += 1; h2c[hi] = ci
            elif op == "D":      ci += 1
            elif op == "I":      hi += 1
        slots = []   # (hyp_index, base, canon_tone_int, current_tone_int, pooled_posterior[6])
        for i, ph in enumerate(hyp_ph):
            b, t = base_tone(ph)
            if t is None: continue
            k = h2c.get(i)
            if k is None or k >= len(spans): continue
            cb, ct = base_tone(row["canon_ph"][k])
            if ct is None: continue
            s, e, _ = spans[k]
            if s is None: continue
            seg = tps[s:e+1][:, TONE_COLS]
            if seg.shape[0] == 0: continue
            slots.append((i, b, int(ct), int(t), seg.mean(0) + 1e-8))
        cache.append(dict(id=row["id"], canon_ph=row["canon_ph"], trans_ph=row["trans_ph"],
                          hyp_ph=hyp_ph, slots=slots))
    return cache

def decode_tone(item, beta, margin):
    hyp = list(item["hyp_ph"])
    for i, b, ct, cur, pooled in item["slots"]:
        w = pooled * (PRIOR_MAT[ct] ** beta)        # diagnosis prior keyed on canonical tone
        new = int(w.argmax())
        if new != cur and (w[new] - w[cur]) >= margin:   # confidence gate
            hyp[i] = f"{b}-{new}"
    return hyp

def score_at(cache, beta, m):
    recs = [dict(canon_ph=c["canon_ph"], trans_ph=c["trans_ph"],
                 hyp_ph=decode_tone(c, beta, m)) for c in cache]
    return score_mdd(recs, verbose=False)

cache = cache_tone(model, test_df, TEST_ROOT)
BETAS  = [0.0, 0.5, 1.0, 2.0]
MARGINS = [0.0, 0.1, 0.2, 0.3]
grid = []
print("beta x margin sweep (F1 / DER / SCORE):")
for beta in BETAS:
    for m in MARGINS:
        a = score_at(cache, beta, m); grid.append((a["score"], beta, m))
        print(f"  beta={beta:>4.2f} m={m:>3}  F1={a['f1']:.4f} R={a['recall']:.3f} "
              f"P={a['precision']:.3f} DER={a['der']:.4f} PER={a['per']:.4f} SCORE={a['score']:.4f}")
b = max(grid); print(f"public-test best: score={b[0]:.4f} beta={b[1]} m={b[2]} | (beta=0,m=0 == pure span override)")

# robustness: 718 dev + HIDDEN final test -> bootstrap the (beta,m) argmax.
import collections
BM = [(beta, m) for beta in BETAS for m in MARGINS]
rng = np.random.default_rng(0); N = len(cache); picks = []
for _ in range(200):
    idx = rng.integers(0, N, N); sub = [cache[i] for i in idx]
    picks.append(max(range(len(BM)), key=lambda j: score_at(sub, BM[j][0], BM[j][1])["score"]))
j = collections.Counter(picks).most_common(1)[0][0]
CH_BETA, CH_M = BM[j]
print(f"bootstrap modal config: beta={CH_BETA} m={CH_M}")
subs = [dict(id=c["id"], predict=" ".join(decode_tone(c, CH_BETA, CH_M))) for c in cache]
pd.DataFrame(subs, columns=["id", "predict"]).to_csv(WORK_DIR / "submission_exp_tone.csv", index=False)
print(f"wrote submission_exp_tone.csv at beta={CH_BETA} m={CH_M}")

In [ ]:
# ============================================================================
# CELL PRIVATE - run the exp_tone pipeline on the PRIVATE (final-ranking) test
# and emit the THREE submissions. Requires the trained `model` + cache_tone /
# decode_tone / PRIOR_MAT / CH_BETA / CH_M from CELL TONE above.
#
# The private metadata ships id,path,canonical(PHONEMES) and NO transcript, so
# it CANNOT be scored locally. Submission format (README_submission.md): columns
# id,path,predict, the SAME 856 rows/order as private_test_submission.csv, the
# file named results.csv, zipped, uploaded to AIHub (one zip per submission).
#
# Three bets across the detection-aggressiveness spectrum (we can't measure the
# private distribution, so spread risk). On the PUBLIC test (official evaluate.py):
#   - copy-canonical (do nothing) = 0.4971  <- the floor; DER defaults to 0
#   - exp_tone full               = 0.5059  <- beats the floor (DER 0.43)
# The private test is 79% long sentences (vs 67% public) -> the long-domain FP
# risk is higher, so the floor and the short-only hedge matter more here.
#   sub1 exp_tone full       (most aggressive: detect everywhere)
#   sub2 copy-canonical      (the 0.497 floor: detect nothing)
#   sub3 exp_tone short-only  (detect short minimal-pairs, revert long sentences)
# ============================================================================
import pandas as pd, glob, zipfile
from pathlib import Path

# locate the private test folder (auto-find under /kaggle/input; else local)
_cands = ["private-test/MDD-Challenge-2025-private-test",
          "/kaggle/input/mdd-private-test/MDD-Challenge-2025-private-test"]
_cands += [str(Path(p).parent.parent)
           for p in glob.glob("/kaggle/input/**/private_test_submission.csv", recursive=True)]
PRIVATE_ROOT = next((Path(c) for c in _cands
                     if (Path(c) / "metadata" / "private_test_submission.csv").exists()), None)
assert PRIVATE_ROOT is not None, "set PRIVATE_ROOT to the dir holding metadata/ and audio_data/"
print("private root:", PRIVATE_ROOT)

priv = pd.read_csv(PRIVATE_ROOT / "metadata" / "private_test_submission.csv")
priv["canon_ph"]  = priv["canonical"].apply(tokenize_phones)
priv["canon_ids"] = priv["canon_ph"].apply(lambda seq: [PHONE2ID.get(p, BLANK_ID) for p in seq])
priv["trans_ph"]  = priv["canon_ph"]            # dummy: decode/score never read it on private
_oov = sorted({p for seq in priv["canon_ph"] for p in seq if p not in PHONE2ID})
n_short = int((priv["canon_ph"].apply(len) <= 6).sum())
print(f"private: {len(priv)} utts | short(<=6)={n_short} long(>6)={len(priv)-n_short} | OOV phones={_oov}")

BETA = globals().get("CH_BETA", 0.0); M = globals().get("CH_M", 0.0)
print(f"decoding private with the bootstrap-chosen operating point: beta={BETA} m={M}")

pcache = cache_tone(model, priv, PRIVATE_ROOT)          # cache in priv row order
preds_full = [decode_tone(c, BETA, M) for c in pcache]  # aligned to priv.iloc[i]
assert len(preds_full) == len(priv), "cache/priv length mismatch"

def emit(name, choose):
    rows = []; changed = 0
    for i in range(len(priv)):
        r = priv.iloc[i]; canon = list(r["canon_ph"]); pred = choose(i, canon)
        changed += int(pred != canon)
        rows.append(dict(id=r["id"], path=r["path"], predict=" ".join(pred)))
    df = pd.DataFrame(rows, columns=["id", "path", "predict"])
    assert len(df) == len(priv) and list(df.columns) == ["id", "path", "predict"]
    csv = Path(WORK_DIR) / (name + "_results.csv"); df.to_csv(csv, index=False, encoding="utf-8")
    zp = Path(WORK_DIR) / (name + ".zip")
    with zipfile.ZipFile(zp, "w", zipfile.ZIP_DEFLATED) as z:
        z.write(csv, arcname="results.csv")             # the inner file MUST be results.csv
    print(f"  {name:24s}: {len(df)} rows | {changed:4d} utts != canonical | -> {zp.name} (results.csv)")
    return df

print("writing the 3 private submissions (each zip contains results.csv):")
emit("sub1_exptone_full",    lambda i, canon: preds_full[i])
emit("sub2_copy_canonical",  lambda i, canon: canon)
emit("sub3_exptone_short",   lambda i, canon: preds_full[i] if len(canon) <= 6 else canon)
print("DONE. Download each .zip from /kaggle/working and submit on AIHub (one per submission).")